> **Superseded:** use `notebooks/build_source_holdout.ipynb` (writes `data/splits/source_holdout/`).

# Uniform source-holdout corpus

Builds a **leave-one-source-out** evaluation set with **matched test sizes**.

| Rule | Choice in this notebook |
|------|-------------------------|
| Attacks from | Full raw / cleaned files (not the old 50-row samples) |
| Holdout | One attack source absent from train/val (attacks + its native benign) |
| Near-duplicates | MiniLM cosine > 0.9 across sources, before sampling |
| Uniformity | Every four-source fold gets the **same** `#attacks` and `#benign` in test |
| Shared benign | Sampled once; never appears in any train/val |
| Fifth dataset | Neuralchemy, capped to the same attack/benign counts |

Output: `data/splits/source_holdout_uniform/<fold>/{train,val,test}.csv`

Simsonsun uses Hugging Face **Dataset_2** (`SimsonsunJailbreakPrompts_Dataset2.csv`, ~2307 unique). Matched attack count defaults to **200**.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.splits.source_holdout import (
    ATTACK_SOURCES,
    EXTERNAL_SOURCE,
    SOURCE_BENIGN,
    VAL_FRACTION,
    _fit_split,
    _norm,
    _with_ids,
    assign_exclusive_source,
    claim,
    drop_near_duplicates,
    load_bravansky_gallery,
    load_bravansky_test_only,
    load_chatgpt_benign,
    load_mpdd,
    load_prompt_injection,
    load_simsonsun,
    write_fold,
)
from src.config import EXTERNAL_TEST_MAIN_CSV, RANDOM_SEED

OUT = ROOT / "data" / "splits" / "source_holdout_uniform"
COLUMNS = ["id", "prompt", "label", "source"]

print("ROOT:", ROOT)
print("OUT :", OUT)

## 1. Uniformity knobs

Change these if you want a different matched size.  
`N_ATTACK_TEST` cannot exceed the smallest source after cleaning (MPDD ≈ 263, prompt-injection ≈ 249, Bravansky pool, Simsonsun Dataset_2 ≫ 200).


In [ ]:
N_ATTACK_TEST = 200         # matched injection count per four-source fold
N_BENIGN_TEST = 200         # matched benign count → 400 prompts total per fold
N_ATTACK_TRAIN_PER_SOURCE = 200  # cap train/val attacks per source
N_NEURALCHEMY_ATTACK = 200  # same attack n for the fifth dataset
N_NEURALCHEMY_BENIGN = 200  # same benign n for the fifth dataset
BRAVANSKY_POOL = 250        # raw Bravansky pool before uniform subsample
SEED = RANDOM_SEED

assert N_ATTACK_TEST > 0 and N_BENIGN_TEST > 0
assert N_ATTACK_TRAIN_PER_SOURCE > 0
print(
    f"Four-source test shape: {N_ATTACK_TEST} attacks + {N_BENIGN_TEST} benign "
    f"= {N_ATTACK_TEST + N_BENIGN_TEST} prompts"
)
print(
    f"Neuralchemy test shape: {N_NEURALCHEMY_ATTACK} attacks + {N_NEURALCHEMY_BENIGN} benign "
    f"= {N_NEURALCHEMY_ATTACK + N_NEURALCHEMY_BENIGN} prompts"
)


## 2. Load full sources (raw)

In [ ]:
simsonsun = load_simsonsun()
mpdd = load_mpdd()
pi_attacks, pi_benign = load_prompt_injection()
chatgpt = load_chatgpt_benign()
bravansky_gallery = load_bravansky_gallery()

inventory = pd.DataFrame(
    [
        {"source": "simsonsun", "role": "attack", "n": len(simsonsun)},
        {"source": "mpdd", "role": "attack", "n": len(mpdd)},
        {"source": "prompt_injection", "role": "attack", "n": len(pi_attacks)},
        {"source": "bravansky_gallery", "role": "attack", "n": len(bravansky_gallery)},
        {"source": "prompt_injection_benign", "role": "benign", "n": len(pi_benign)},
        {"source": "chatgpt_benign", "role": "benign", "n": len(chatgpt)},
    ]
)
display(inventory)

fig, ax = plt.subplots(figsize=(8, 3.2))
colors = ["#c44e52" if r == "attack" else "#4c72b0" for r in inventory["role"]]
ax.barh(inventory["source"], inventory["n"], color=colors)
ax.set_xlabel("prompts")
ax.set_title("Raw / cleaned source sizes (before exclusivity & near-dup)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Exact exclusivity + MiniLM near-duplicates

In [ ]:
attacks, taken, exact_overlaps = assign_exclusive_source(
    [
        ("simsonsun", simsonsun),
        ("bravansky", bravansky_gallery),
        ("mpdd", mpdd),
        ("prompt_injection", pi_attacks),
    ]
)
attacks, near_dup_report = drop_near_duplicates(attacks)

taken = set()
for frame in attacks.values():
    taken.update(frame["prompt_key"].tolist())

extra = load_bravansky_test_only(
    set(attacks["bravansky"]["prompt_key"]), taken, BRAVANSKY_POOL
)
extra, bravansky_extra_overlap = claim(extra, taken)
bravansky_pool = pd.concat([attacks["bravansky"], extra], ignore_index=True)

after = pd.DataFrame(
    [
        {"source": name, "n_attacks": len(attacks[name] if name != "bravansky" else bravansky_pool)}
        for name in ATTACK_SOURCES
    ]
)
# bravansky row should use pool size
after.loc[after["source"] == "bravansky", "n_attacks"] = len(bravansky_pool)

display(after)
print("Exact overlaps removed:", exact_overlaps)
print(
    "Near-duplicates removed:",
    near_dup_report["removed"],
    near_dup_report["removed_by_source"],
)
print("Smallest attack pool:", int(after["n_attacks"].min()))
assert int(after["n_attacks"].min()) >= N_ATTACK_TEST, (
    f"N_ATTACK_TEST={N_ATTACK_TEST} exceeds smallest source "
    f"({int(after['n_attacks'].min())}). Lower the knob."
)

## 4. Shared benign pool (test-only)

In [ ]:
benign_all = pd.concat([chatgpt, pi_benign], ignore_index=True)
benign_all = benign_all.drop_duplicates(subset=["prompt_key"]).reset_index(drop=True)
benign_all = benign_all[~benign_all["prompt_key"].isin(taken)].reset_index(drop=True)

assert len(benign_all) >= N_BENIGN_TEST + 40, (
    f"Need enough benign for test ({N_BENIGN_TEST}) plus train leftovers; have {len(benign_all)}"
)

shared_benign, benign_fit = train_test_split(
    benign_all,
    train_size=N_BENIGN_TEST,
    random_state=SEED,
    stratify=benign_all["source"] if benign_all["source"].nunique() > 1 else None,
)
shared_benign = shared_benign.reset_index(drop=True)
benign_fit = benign_fit.reset_index(drop=True)
shared_keys = set(shared_benign["prompt_key"])

print("Shared benign test:", len(shared_benign))
print(shared_benign["source"].value_counts().to_dict())
print("Benign fit pool (may enter train/val):", len(benign_fit))
print(benign_fit["source"].value_counts().to_dict())

## 5. Helpers — uniform subsample + fold assembly

In [ ]:
def sample_n(frame: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    if len(frame) < n:
        raise ValueError(f"Need {n} rows, have {len(frame)}")
    return frame.sample(n=n, random_state=seed).reset_index(drop=True)


def exclude_source_benign(benign_fit_df: pd.DataFrame, held_out: str) -> pd.DataFrame:
    banned = SOURCE_BENIGN.get(held_out, ())
    if not banned:
        return benign_fit_df
    return benign_fit_df[~benign_fit_df["source"].isin(banned)].reset_index(drop=True)


def attack_pool(held_out: str) -> pd.DataFrame:
    if held_out == "bravansky":
        return bravansky_pool
    return attacks[held_out]


def train_attack_slice(name: str, seed: int) -> pd.DataFrame:
    """Subsample a source for train/val. Bravansky train stays on the gallery only."""
    frame = attacks[name]
    n = min(N_ATTACK_TRAIN_PER_SOURCE, len(frame))
    return sample_n(frame, n, seed)


def build_four_source_fold(held_out: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Train on other three sources (capped); uniform test = N_ATTACK_TEST + N_BENIGN_TEST."""
    test_attacks = sample_n(attack_pool(held_out), N_ATTACK_TEST, SEED + hash(held_out) % 10_000)
    train_attacks = pd.concat(
        [
            train_attack_slice(name, SEED + 1000 + i)
            for i, name in enumerate(ATTACK_SOURCES)
            if name != held_out
        ],
        ignore_index=True,
    )

    fit_benign = exclude_source_benign(benign_fit, held_out)
    train, val = _fit_split(pd.concat([train_attacks, fit_benign], ignore_index=True))

    # Uniform benign test: always the same shared_benign sample (already size N_BENIGN_TEST).
    test = pd.concat([test_attacks, shared_benign], ignore_index=True)

    # Leak checks
    seen = {_norm(p) for p in list(train["prompt"]) + list(val["prompt"])}
    leak = sum(_norm(p) in seen for p in test["prompt"])
    assert leak == 0, f"{held_out}: {leak} test prompts leak into train/val"
    assert (test["label"] == "injection").sum() == N_ATTACK_TEST
    assert (test["label"] == "benign").sum() == N_BENIGN_TEST
    return train, val, test


def build_neuralchemy_fold() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, int]:
    neuralchemy = pd.read_csv(EXTERNAL_TEST_MAIN_CSV)
    neuralchemy["prompt_key"] = neuralchemy["prompt"].map(_norm)
    dropped = int(neuralchemy["prompt_key"].isin(taken).sum())
    neuralchemy = neuralchemy[~neuralchemy["prompt_key"].isin(taken)].copy()
    neuralchemy["source"] = "neuralchemy:" + neuralchemy["source"].astype(str)

    inj = neuralchemy[neuralchemy["label"] == "injection"]
    ben = neuralchemy[neuralchemy["label"] == "benign"]
    test_inj = sample_n(inj, N_NEURALCHEMY_ATTACK, SEED + 7)
    test_ben = sample_n(ben, N_NEURALCHEMY_BENIGN, SEED + 11)
    test = pd.concat([test_inj, test_ben], ignore_index=True)

    train_attacks = pd.concat(
        [train_attack_slice(name, SEED + 2000 + i) for i, name in enumerate(ATTACK_SOURCES)],
        ignore_index=True,
    )
    pool = pd.concat([train_attacks, benign_fit], ignore_index=True)
    train, val = _fit_split(pool)

    seen = {_norm(p) for p in list(train["prompt"]) + list(val["prompt"])}
    leak = sum(_norm(p) in seen for p in test["prompt"])
    assert leak == 0
    return train, val, test, dropped


## 6. Build all folds and write CSVs

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
summary_rows = []
fold_payload = {}

for held in ATTACK_SOURCES:
    train, val, test = build_four_source_fold(held)
    write_fold(OUT / held, train, val, test)
    fold_payload[held] = {
        "train": len(train),
        "val": len(val),
        "test": len(test),
        "test_injection": int((test["label"] == "injection").sum()),
        "test_benign": int((test["label"] == "benign").sum()),
        "train_benign_by_source": train.loc[train["label"] == "benign", "source"]
        .value_counts()
        .to_dict(),
    }
    summary_rows.append(
        {
            "fold": held,
            "train": len(train),
            "val": len(val),
            "test": len(test),
            "test_injection": fold_payload[held]["test_injection"],
            "test_benign": fold_payload[held]["test_benign"],
        }
    )
    print(
        f"{held:18s} train={len(train):4d} val={len(val):3d} "
        f"test={len(test):3d} (inj={fold_payload[held]['test_injection']}, "
        f"ben={fold_payload[held]['test_benign']})"
    )

train, val, test, dropped = build_neuralchemy_fold()
write_fold(OUT / EXTERNAL_SOURCE, train, val, test)
fold_payload[EXTERNAL_SOURCE] = {
    "train": len(train),
    "val": len(val),
    "test": len(test),
    "test_injection": int((test["label"] == "injection").sum()),
    "test_benign": int((test["label"] == "benign").sum()),
    "dropped_overlap_with_four_sources": dropped,
    "train_benign_by_source": train.loc[train["label"] == "benign", "source"]
    .value_counts()
    .to_dict(),
}
summary_rows.append(
    {
        "fold": EXTERNAL_SOURCE,
        "train": len(train),
        "val": len(val),
        "test": len(test),
        "test_injection": fold_payload[EXTERNAL_SOURCE]["test_injection"],
        "test_benign": fold_payload[EXTERNAL_SOURCE]["test_benign"],
    }
)
print(
    f"{EXTERNAL_SOURCE:18s} train={len(train):4d} val={len(val):3d} "
    f"test={len(test):3d} (inj={fold_payload[EXTERNAL_SOURCE]['test_injection']}, "
    f"ben={fold_payload[EXTERNAL_SOURCE]['test_benign']})"
)

summary = pd.DataFrame(summary_rows)
display(summary)

## 7. Uniformity check + figure

In [ ]:
four = summary[summary["fold"].isin(ATTACK_SOURCES)]
assert four["test_injection"].nunique() == 1, four["test_injection"].tolist()
assert four["test_benign"].nunique() == 1, four["test_benign"].tolist()
assert four["test"].nunique() == 1, four["test"].tolist()
print("Four-source tests are uniform:", four["test"].iloc[0], "prompts each.")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
axes[0].bar(summary["fold"], summary["test_injection"], color="#c44e52")
axes[0].set_title("Test injections (uniform)")
axes[0].tick_params(axis="x", rotation=30)
axes[0].set_ylabel("count")
axes[1].bar(summary["fold"], summary["test_benign"], color="#4c72b0")
axes[1].set_title("Test benign (uniform)")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
fig_path = OUT / "uniform_test_sizes.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
print("Saved", fig_path)
plt.show()

# Train benign inventory (mentor asked what each fold trains on)
benign_docs = []
for fold, payload in fold_payload.items():
    benign_docs.append({"fold": fold, **{f"train_{k}": v for k, v in payload.get("train_benign_by_source", {}).items()}})
display(pd.DataFrame(benign_docs).fillna(0).astype({c: int for c in []}, errors="ignore"))

## 8. Manifest

In [ ]:
manifest = {
    "seed": SEED,
    "protocol": "uniform source-holdout",
    "output_dir": str(OUT.relative_to(ROOT)).replace("\\", "/"),
    "uniformity": {
        "n_attack_test": N_ATTACK_TEST,
        "n_benign_test": N_BENIGN_TEST,
        "n_attack_train_per_source": N_ATTACK_TRAIN_PER_SOURCE,
        "four_source_test_total": N_ATTACK_TEST + N_BENIGN_TEST,
        "neuralchemy_attack": N_NEURALCHEMY_ATTACK,
        "neuralchemy_benign": N_NEURALCHEMY_BENIGN,
        "note": (
            "Test uses 200+200 matched counts. Dataset_2 is only a sampling pool; "
            "train/val attacks are capped at N_ATTACK_TRAIN_PER_SOURCE per source."
        ),
    },
    "near_duplicate_filter": near_dup_report,
    "exact_overlaps": exact_overlaps,
    "bravansky_extra_overlap": bravansky_extra_overlap,
    "shared_benign_test": {
        "rows": int(len(shared_benign)),
        "by_source": shared_benign["source"].value_counts().to_dict(),
        "note": "Identical benign test set for every four-source fold; never in train/val.",
    },
    "folds": fold_payload,
    "how_to_score": [
        "Point scripts/eval_loo_holdout.py LOO path at data/splits/source_holdout_uniform, or",
        "copy/symlink this folder over data/splits/source_holdout after review.",
    ],
}

(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
(OUT / "near_duplicate_spot_check.json").write_text(
    json.dumps(near_dup_report.get("spot_check", []), indent=2) + "\n", encoding="utf-8"
)
summary.to_csv(OUT / "summary_table.csv", index=False)
print("Wrote", OUT / "manifest.json")
print("Wrote", OUT / "summary_table.csv")


## 9. Peek at one fold

In [ ]:
example = pd.read_csv(OUT / "mpdd" / "test.csv")
print(example["label"].value_counts().to_dict())
print(example["source"].value_counts().to_dict())
display(example.head(3))
display(example.tail(3))